In [1]:
import numpy as np 
import pandas as pd 
pd.options.plotting.backend = "plotly"
import random
from glob import glob
import os 
import shutil 
from tqdm import tqdm 
tqdm.pandas()
import copy 
import joblib 
from collections import defaultdict
import gc 
import matplotlib.pyplot as plt 
from matplotlib.patches import Rectangle

In [2]:
import cv2 
from sklearn.model_selection import (StratifiedKFold,
                                     KFold,
                                     StratifiedGroupKFold)
import torch 
from torch import nn 
from torch.utils.data import Dataset,DataLoader
from torch.cuda import amp 
import timm 
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio
from joblib import Parallel, delayed
from colorama import Fore, Back, Style
c_  = Fore.GREEN
sr_ = Style.RESET_ALL

import dagshub 
import mlflow


c:\Users\rsurs\OneDrive\Documents\medseg-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pydantic import BaseModel
from typing import List

class Config(BaseModel):
    seed: int = 101
    debug: bool = False
    comment: str = "unet-efficientnet_b1-224x224-aug2-split2"
    model_name: str = "Unet"
    backbone: str = "efficientnet-b1"
    train_bs: int = 128
    valid_bs: int = train_bs * 2 
    img_size: List[int] = [224,224]
    epochs: int = 15
    lr: float = 2e-3
    scheduler: str = "CosineAnnealingLR"
    min_lr: float = 1e-6
    T_max: int = int(30000/train_bs*epochs) + 50
    T_0: int = 25
    warmpup_epochs: int = 0
    weight_decay: int = 1e-6
    n_accumulate: int = max(1,32//train_bs)
    n_fold: int = 5 
    num_classes: int = 3 
    device: str = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

In [6]:
cfg = Config()

#### Reproducibility

In [8]:
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False 

    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(cfg.seed)

### Meta-Data